In [1]:
import pandas as pd
import numpy as np
import os

# Detect desktop path
if os.name == 'nt':  # Windows
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
else:  # macOS or Linux
    desktop = os.path.join(os.path.expanduser("~"), "Desktop")

# Normalized data from Table 1
data = {
    "Alternatives": ["REF", "PG5", "PG10", "PG15", "PG20", "MW5", "MW10", "MW15", "MW20",
                     "SW5", "SW10", "SW15", "SW20", "CSW5", "CSW10", "CSW15", "CSW20"],
    "Compressive strength": [0.95, 0.93, 0.60, 0.19, 0.00, 1.00, 0.68, 0.65, 0.44,
                             0.84, 0.84, 0.69, 0.52, 0.93, 0.86, 0.70, 0.66],
    "Rate of absorption of water": [0.44, 0.58, 0.76, 0.89, 1.00, 0.23, 0.17, 0.05, 0.02,
                                    0.32, 0.07, 0.00, 0.01, 0.53, 0.51, 0.49, 0.49],
    "Open porosity": [0.46, 0.51, 0.72, 0.91, 1.00, 0.02, 0.01, 0.00, 0.06,
                      0.47, 0.48, 0.51, 0.57, 0.50, 0.55, 0.58, 0.62],
    "Electrical resistivity": [1.00, 0.95, 0.90, 0.81, 0.74, 0.95, 0.92, 0.80, 0.71,
                               0.94, 0.94, 0.70, 0.52, 0.90, 0.88, 0.38, 0.00],
    "Global Warming Potential": [1.00, 0.69, 0.39, 0.33, 0.03, 0.69, 0.39, 0.34, 0.04,
                                 0.68, 0.37, 0.31, 0.00, 0.72, 0.44, 0.42, 0.14],
    "Eco-efficiency (ISO 14045)": [0.83, 0.93, 0.64, 0.15, 0.00, 1.00, 0.74, 0.72, 0.56,
                                   0.82, 0.93, 0.78, 0.68, 0.91, 0.93, 0.75, 0.80],
    "Environmental life cycle costing": [0.00, 0.26, 0.25, 0.25, 0.25, 0.29, 0.28, 0.28, 0.27,
                                         0.14, 0.13, 0.13, 0.12, 0.55, 0.74, 0.87, 1.00],
    "Circular Economy": [0.63, 0.71, 0.80, 0.88, 0.97, 0.71, 0.80, 0.88, 0.99,
                         0.71, 0.81, 0.89, 1.00, 0.64, 0.48, 0.27, 0.00]
}

df = pd.DataFrame(data)
df.set_index("Alternatives", inplace=True)

# Weights (converted from percentage to fraction)
weights = {
    "Compressive strength": 21.2 / 100,
    "Open porosity": 5.6 / 100,
    "Rate of absorption of water": 7.9 / 100,
    "Electrical resistivity": 10.3 / 100,
    "Global Warming Potential": 18.0 / 100,
    "Eco-efficiency (ISO 14045)": 17.1 / 100,
    "Environmental life cycle costing": 12.9 / 100,
    "Circular Economy": 7.2 / 100
}

# Define cost (False) and benefit (True) criteria
is_benefit = {
    "Compressive strength": True,
    "Open porosity": False,          # cost
    "Rate of absorption of water": False,  # cost
    "Electrical resistivity": True,
    "Global Warming Potential": False,     # cost
    "Eco-efficiency (ISO 14045)": True,
    "Environmental life cycle costing": True,
    "Circular Economy": True
}

# Convert cost criteria to benefit (1 - value)
df_adjusted = df.copy()
for col in df.columns:
    if not is_benefit[col]:
        df_adjusted[col] = 1 - df[col]

# Vector normalization (r_ij = x_ij / sqrt(sum(x_ij²)))
df_normalized = df_adjusted.copy()
for col in df_normalized.columns:
    norm = np.sqrt(np.sum(df_normalized[col] ** 2))
    if norm != 0:
        df_normalized[col] = df_normalized[col] / norm

# Apply weights
df_weighted = df_normalized.copy()
for col in df_weighted.columns:
    df_weighted[col] = df_weighted[col] * weights[col]

# Ideal (A+) and anti-ideal (A-) solutions
A_plus = df_weighted.max()
A_minus = df_weighted.min()

# Euclidean distances
D_plus = np.sqrt(((df_weighted - A_plus) ** 2).sum(axis=1))
D_minus = np.sqrt(((df_weighted - A_minus) ** 2).sum(axis=1))

# Relative closeness index (C_i)
C_i = D_minus / (D_plus + D_minus)

# Create results DataFrame
results = pd.DataFrame({
    "D+": D_plus,
    "D-": D_minus,
    "C_i": C_i
})
results = results.sort_values("C_i", ascending=False)

# Full path to save on desktop
output_path = os.path.join(desktop, "TOPSIS_Results.xlsx")

# Export to Excel with multiple sheets
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Normalized Data")
    pd.Series(weights).to_frame(name="Weight (fraction)").to_excel(writer, sheet_name="Weights")
    df_adjusted.to_excel(writer, sheet_name="Adjusted Data (Benefit)")
    df_weighted.to_excel(writer, sheet_name="Weighted Normalized Matrix")
    results.to_excel(writer, sheet_name="TOPSIS Results")

print(f"✅ File successfully saved at:\n{output_path}")

✅ File successfully saved at:
C:\Users\Bianca\Desktop\TOPSIS_Results.xlsx
